# 10. Clustering y Reducción de Dimensionalidad

En este notebook aprenderás técnicas de clustering (agrupamiento) y reducción de dimensionalidad, con ejemplos prácticos usando scikit-learn.

## Objetivo
- Comprender la teoría y aplicaciones de clustering y reducción de dimensionalidad.
- Implementar K-Means, DBSCAN, PCA y t-SNE.
- Usar el método del codo para elegir K.
- Evaluar clustering con métricas cuantitativas.

## Prerequisitos

> 📌 **Prerequisitos:** Haber completado los notebooks [01](./01_intro_machine_learning.ipynb) y [02](./02_preprocesamiento_visualizacion.ipynb).

- Conceptos de escalado, PCA, y evaluación de modelos.

## 1. Introducción teórica

| Técnica | Tipo | Ventaja | Desventaja |
|---------|------|---------|------------|
| **K-Means** | Clustering | Rápido, escalable | Requiere K, clusters esféricos |
| **DBSCAN** | Clustering | Forma libre, detecta outliers | Sensible a eps y min_samples |
| **PCA** | Reducción | Lineal, rápido, preserva varianza | Solo relaciones lineales |
| **t-SNE** | Reducción | Preserva estructura local | Solo para visualización, lento |

## 2. Importación de librerías

In [ ]:
# === Reproducibilidad ===
import random
import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score

import warnings
warnings.filterwarnings('ignore')

sns.set(style="whitegrid")
%matplotlib inline

## 3. Carga y exploración del dataset

In [ ]:
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['target'] = iris.target
df.head()

In [ ]:
sns.pairplot(df, hue='target', palette='Set1')
plt.suptitle('Distribución de clases reales', y=1.02)
plt.show()

## 4. Preprocesamiento

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[iris.feature_names])
print('Media:', X_scaled.mean(axis=0).round(2))
print('Desviación estándar:', X_scaled.std(axis=0).round(2))

## 5. Método del Codo para elegir K

El método del codo grafica la inercia (suma de distancias al centroide) vs K, buscando el punto donde la reducción de inercia se desacelera.

In [ ]:
inertias = []
sil_scores = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    inertias.append(kmeans.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(K_range, inertias, 'bo-')
axes[0].set_xlabel('K (número de clusters)')
axes[0].set_ylabel('Inercia')
axes[0].set_title('Método del Codo')
axes[0].axvline(x=3, color='r', linestyle='--', alpha=0.5, label='K=3')
axes[0].legend()

axes[1].plot(K_range, sil_scores, 'go-')
axes[1].set_xlabel('K (número de clusters)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette vs K')
axes[1].axvline(x=3, color='r', linestyle='--', alpha=0.5, label='K=3')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Clustering: K-Means

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

kmeans = KMeans(n_clusters=3, random_state=SEED, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=clusters, palette='Set2', s=60, ax=axes[0])
axes[0].set_title('Clusters K-Means (PCA 2D)')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')

sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=df['target'], palette='Set1', s=60, ax=axes[1])
axes[1].set_title('Clases reales (PCA 2D)')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')

plt.tight_layout()
plt.show()

print(f'Silhouette Score: {silhouette_score(X_scaled, clusters):.2f}')
print(f'Adjusted Rand Index: {adjusted_rand_score(df["target"], clusters):.2f}')

## 7. Clustering: DBSCAN

In [ ]:
dbscan = DBSCAN(eps=0.6, min_samples=5)
db_clusters = dbscan.fit_predict(X_scaled)

plt.figure(figsize=(8,6))
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=db_clusters, palette='Set2', s=60)
plt.title('Clusters DBSCAN (PCA 2D)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.show()

n_clusters = len(set(db_clusters)) - (1 if -1 in db_clusters else 0)
n_noise = list(db_clusters).count(-1)
print(f'Clusters encontrados: {n_clusters}, Outliers: {n_noise}')

if len(set(db_clusters)) > 1:
    print(f'Silhouette Score: {silhouette_score(X_scaled, db_clusters):.2f}')
    print(f'Adjusted Rand Index: {adjusted_rand_score(df["target"], db_clusters):.2f}')

## 8. Reducción de dimensionalidad: t-SNE

In [ ]:
tsne = TSNE(n_components=2, random_state=SEED, n_iter=1000, perplexity=30)
X_tsne = tsne.fit_transform(X_scaled)

plt.figure(figsize=(8,6))
sns.scatterplot(x=X_tsne[:,0], y=X_tsne[:,1], hue=df['target'], palette='Set1', s=60)
plt.title('Visualización t-SNE de las clases reales')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.show()

## 9. Discusión y Conclusiones

- **K-Means** funciona bien cuando los clusters son esféricos y de tamaño similar.
- **DBSCAN** detecta formas arbitrarias y outliers, pero requiere ajustar `eps` y `min_samples`.
- **PCA** y **t-SNE** son complementarios: PCA para reducción lineal, t-SNE solo para visualización.
- El método del codo y Silhouette Score ayudan a elegir el número de clusters.
- Siempre escalar los datos antes de clustering.

## 10. Ejercicios Propuestos

1. **Ejercicio 1:** Aplica K-Means y DBSCAN al dataset `load_digits()`. ¿Cuántos clusters encuentra DBSCAN?

2. **Ejercicio 2:** Varía `eps` de DBSCAN entre 0.3 y 1.0 y grafica el número de clusters vs eps.

3. **Ejercicio 3:** Compara PCA y t-SNE en el dataset Digits. ¿Cuál separa mejor las clases visualmente?

4. **Ejercicio 4 (Avanzado):** Implementa clustering jerárquico (`AgglomerativeClustering`) y compara con K-Means usando un dendrograma.

## 11. Referencias y Recursos

- [Scikit-learn: Clustering](https://scikit-learn.org/stable/modules/clustering.html)
- [Scikit-learn: Dimensionality Reduction](https://scikit-learn.org/stable/modules/unsupervised_reduction.html)
- [How to Use t-SNE Effectively](https://distill.pub/2016/misread-tsne/)

---

📎 **Notebook anterior:** [09. Autoencoders](./09_autoencoders.ipynb)  
📎 **Notebook siguiente:** [11. Interpretabilidad de Modelos](./11_interpretabilidad_modelos.ipynb)